# Hotel Booking Cancellation Prediction

**Classification Project 16 of 100** - Predict whether a hotel booking will be cancelled.

End-to-end workflow: EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.

## 2. Project Overview

Hotel cancellations cost the hospitality industry billions annually. This notebook builds a binary classifier predicting whether a booking will be cancelled (is_canceled=1) using the Hotel Booking Demand dataset (~119k rows, 32 features).

**Workflow:** Download -> EDA -> preprocess -> baselines -> LazyPredict -> FLAML -> top 3 -> error analysis.

## 3. Learning Objectives

1. Work with a rich hospitality dataset (~119k bookings, 32 features)
2. Handle high-cardinality categoricals (country, agent, company)
3. Detect and remove target leakage (reservation_status)
4. Engineer lead-time and guest-based features
5. Build leakage-free ColumnTransformer pipelines
6. Evaluate with accuracy, F1, ROC-AUC, confusion matrix
7. Benchmark with LazyPredict and tune with FLAML
8. Reason about revenue impact of FN vs FP
9. Handle near-balanced classes (~37% cancellation rate)
10. Understand post-booking features that must be excluded

## 4. Problem Statement

> Given booking details (lead time, deposit type, customer type, market segment, previous cancellations, special requests, etc.), predict whether the booking will be cancelled.

Binary classification with ~37% cancelled. F1 and ROC-AUC are primary metrics.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Hotels | Reduce revenue loss from cancellations |
| Revenue managers | Optimise overbooking strategies |
| Operations | Better staff and resource planning |
| Guests | Targeted retention offers |
| ML learners | Leakage detection, mixed features, business evaluation |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Name | Hotel Booking Demand |
| Kaggle slug | jessemostipak/hotel-booking-demand |
| Rows | ~119,390 |
| Features | 32 (numeric + categorical) |
| Target | is_canceled (0=kept, 1=cancelled) |
| Class balance | ~63% kept, ~37% cancelled |

**Leakage warning:** reservation_status and reservation_status_date directly encode the outcome and must be dropped.

## 7. Dataset Source and License Notes

- Original paper: Antonio, de Almeida, Nunes (2019). Hotel booking demand datasets. Data in Brief.
- Kaggle: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
- License: CC0 Public Domain.
- Data from two Portuguese hotels, July 2015 to August 2017.

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib

for pkg in ["lazypredict", "flaml", "kagglehub", "xgboost", "lightgbm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report
)

from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "jessemostipak/hotel-booking-demand"
TARGET = "is_canceled"
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42
FLAML_BUDGET = 120
MAX_ROWS = 50_000
LEAK_COLS = ["reservation_status", "reservation_status_date"]
print(f"Target: {TARGET} | Test: {TEST_SIZE} | Seed: {SEED} | Max rows: {MAX_ROWS}")

## 11. Dataset Download or Loading

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Dataset downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Failed to download: {e}")
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

In [ ]:
print(f"Missing (top 5):")
print(df_raw.isnull().sum().sort_values(ascending=False).head(5))
df = df_raw.copy()
drop = [c for c in LEAK_COLS if c in df.columns]
if drop:
    df = df.drop(columns=drop)
    print(f"Dropped leaking columns: {drop}")
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
assert TARGET in df.columns
print(df[TARGET].value_counts())
print(f"Cancel rate: {df[TARGET].mean():.3f}")
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Subsampled to {MAX_ROWS}")

## 13. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue","salmon"])
ax.set_title("Target Distribution")
ax.set_xticklabels(["Not Cancelled","Cancelled"], rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4))
for ax, col in zip(axes, ["lead_time","adr","total_of_special_requests"]):
    if col in df.columns:
        for lb in [0,1]:
            ax.hist(df[df[TARGET]==lb][col].dropna(), bins=40, alpha=0.5, label="Cancel" if lb else "Keep")
        ax.set_title(col); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
cat_show = [c for c in ["deposit_type","customer_type","market_segment"] if c in df.columns]
fig, axes = plt.subplots(1, len(cat_show), figsize=(5*len(cat_show),4))
if len(cat_show)==1: axes=[axes]
for ax, col in zip(axes, cat_show):
    df.groupby(col)[TARGET].mean().sort_values(ascending=False).plot.bar(ax=ax, color="salmon")
    ax.set_title(f"Cancel Rate by {col}"); ax.set_ylabel("Rate")
plt.tight_layout(); plt.show()

## 14. Target Analysis

~37% cancellation rate - near-balanced. F1 ensures balanced precision/recall.

In [ ]:
cancel_rate = df[TARGET].mean()
print(f"Cancel rate: {cancel_rate:.1%}")
print(f"Majority baseline: {max(cancel_rate, 1-cancel_rate):.1%}")

## 15. Train / Validation / Test Split Strategy

70/15/15 stratified split, before preprocessing.

In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for n, ys in [("Train",y_train),("Val",y_val),("Test",y_test)]: print(f"{n} cancel rate: {ys.mean():.3f}")

## 16. Preprocessing Strategy

- Numeric: median impute + StandardScaler
- Categorical (<=50 unique): most-frequent impute + OneHotEncoder
- High-cardinality (>50 unique): dropped

In [ ]:
high_card = [c for c in X_train.select_dtypes(include=["object"]).columns if X_train[c].nunique()>50]
if high_card:
    print(f"Dropping high-cardinality: {high_card}")
    X_train=X_train.drop(columns=high_card); X_val=X_val.drop(columns=high_card); X_test=X_test.drop(columns=high_card)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat_cols)
], remainder="drop")
print(f"Num: {len(num_cols)}  Cat: {len(cat_cols)}")

## 17. Feature Engineering

- TOTAL_GUESTS, TOTAL_NIGHTS, IS_REPEATED, HAS_SPECIAL_REQUESTS

In [ ]:
def eng(d):
    d = d.copy()
    gc = [c for c in ["adults","children","babies"] if c in d.columns]
    if gc: d["TOTAL_GUESTS"] = d[gc].sum(axis=1)
    nc = [c for c in ["stays_in_weekend_nights","stays_in_week_nights"] if c in d.columns]
    if nc: d["TOTAL_NIGHTS"] = d[nc].sum(axis=1)
    if "previous_bookings_not_canceled" in d.columns: d["IS_REPEATED"] = (d["previous_bookings_not_canceled"]>0).astype(float)
    if "total_of_special_requests" in d.columns: d["HAS_SPECIAL_REQUESTS"] = (d["total_of_special_requests"]>0).astype(float)
    return d
X_train=eng(X_train); X_val=eng(X_val); X_test=eng(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat_cols)
], remainder="drop")
print(f"Total features: {X_train.shape[1]}")

## 18. Baseline Model

In [ ]:
results = {}
for name, clf in [("Dummy",DummyClassifier(strategy="most_frequent",random_state=SEED)),
                  ("LogReg",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED)),
                  ("RF",RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=SEED,n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {"Accuracy":accuracy_score(y_val,yp),"F1":f1_score(y_val,yp),"Recall":recall_score(y_val,yp)}
    if hasattr(clf,"predict_proba"): r["ROC-AUC"]=roc_auc_score(y_val,pipe.predict_proba(X_val)[:,1])
    results[name]=r; print(f"{name}: {r}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
X_train_fl=X_train.copy(); X_val_fl=X_val.copy()
for c in cat_cols: X_train_fl[c]=X_train_fl[c].astype(str); X_val_fl[c]=X_val_fl[c].astype(str)
automl = AutoML()
automl.fit(X_train_fl, y_train, task="classification", metric="f1", time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f"Best: {automl.best_estimator}")
yp_fl=automl.predict(X_val_fl); ypr_fl=automl.predict_proba(X_val_fl)[:,1]
results["FLAML"]={"Accuracy":accuracy_score(y_val,yp_fl),"F1":f1_score(y_val,yp_fl),"ROC-AUC":roc_auc_score(y_val,ypr_fl)}
print(results["FLAML"])

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm=True
except ImportError: has_lgbm=False
try:
    from xgboost import XGBClassifier; has_xgb=True
except ImportError: has_xgb=False
w=(y_train==0).sum()/max((y_train==1).sum(),1)
top3={}
if has_lgbm: top3["LightGBM"]=LGBMClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,class_weight="balanced",random_state=SEED,verbose=-1,n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"]=ExtraTreesClassifier(n_estimators=500,class_weight="balanced",random_state=SEED,n_jobs=-1)
if has_xgb: top3["XGBoost"]=XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,scale_pos_weight=w,random_state=SEED,verbosity=0,n_jobs=-1,eval_metric="logloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"]=AdaBoostClassifier(n_estimators=200,random_state=SEED)
top3["GradBoosting"]=GradientBoostingClassifier(n_estimators=300,learning_rate=0.05,max_depth=5,random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t=preprocessor.fit_transform(X_train); X_te_t=preprocessor.transform(X_test)
final={}
for name,mdl in top3.items():
    mdl.fit(X_tr_t,y_train); yp=mdl.predict(X_te_t); ypr=mdl.predict_proba(X_te_t)[:,1]
    final[name]={"Accuracy":accuracy_score(y_test,yp),"Precision":precision_score(y_test,yp),"Recall":recall_score(y_test,yp),"F1":f1_score(y_test,yp),"ROC-AUC":roc_auc_score(y_test,ypr),"PR-AUC":average_precision_score(y_test,ypr)}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=["Not Cancelled","Cancelled"]))
pd.DataFrame(final).T

In [ ]:
fig,axes=plt.subplots(1,len(top3),figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes,top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_te_t),ax=ax,cmap="Blues",display_labels=["Keep","Cancel"])
    ax.set_title(n)
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m,X_te_t,y_test,ax=axes[0],name=n)
    PrecisionRecallDisplay.from_estimator(m,X_te_t,y_test,ax=axes[1],name=n)
axes[0].plot([0,1],[0,1],"k--",alpha=0.5); axes[0].set_title("ROC")
axes[1].set_title("Precision-Recall")
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
bm=list(top3.values())[0]; ypb=bm.predict(X_te_t)
fn=(y_test==1)&(ypb==0); fp=(y_test==0)&(ypb==1)
print(f"FN: {fn.sum()}  FP: {fp.sum()}")
td=X_test.copy(); td["err"]="correct"
td.loc[fn.values,"err"]="fn"; td.loc[fp.values,"err"]="fp"
for c in ["lead_time","adr","total_of_special_requests"]:
    if c in td.columns: print(f"\n{c}:\n{td.groupby('err')[c].mean()}")

## 24. Interpretation and Business Insight

**Key findings:**
1. Lead time is the strongest predictor
2. Deposit type (Non Refund) drastically reduces cancellations
3. Previous cancellations predict future cancellations
4. Special requests indicate higher commitment

**Revenue recommendations:**
- Require deposits for bookings with lead time > 60 days
- Offer early-bird non-refundable rates
- Flag repeat cancellers for special handling

## 25. Limitations

1. Two hotels only - both in Portugal
2. 2015-2017 data - pre-COVID
3. Country dropped for high cardinality
4. No pricing strategy data
5. Subsampled for speed

## 26. How to Improve This Project

1. Target-encode country instead of dropping it
2. Add temporal features (arrival month, day of week)
3. Try CatBoost for native categorical handling
4. Add SHAP for booking-level explanations
5. Build revenue-impact simulator
6. Use cross-validation

## 27. Production Considerations

- Real-time scoring at reservation time
- Tie predictions to overbooking policy
- Monitor actual vs predicted cancellation rates
- No discrimination by nationality
- Post-COVID retraining needed

## 28. Common Mistakes

1. Including reservation_status - directly leaks the target
2. Not dropping company/agent - high-cardinality noise
3. Applying SMOTE - unnecessary with 37% positive rate
4. Ignoring lead_time - most important predictor
5. Not stratifying splits
6. Using accuracy alone

## 29. Mini Challenge / Exercises

1. Add country back using target encoding
2. Optimal threshold given FN=$150 and FP=$20
3. Split by hotel type - which is easier?
4. Add SHAP - top 5 drivers?
5. Rule-based model with lead_time + deposit_type

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary classification - hotel booking cancellation |
| Dataset | ~119k bookings, 32 features, ~37% cancellation |
| Best metric | F1 and ROC-AUC |
| Baseline | DummyClassifier ~63% accuracy |
| Best models | Gradient boosting family |
| Key insight | Lead time and deposit type dominate |
| Limitation | Two Portuguese hotels, pre-COVID |

**What you learned:**
- Detecting and removing leaking columns
- Feature engineering from booking domain
- Handling high-cardinality categoricals
- Full benchmark -> AutoML -> top 3 pipeline